#모델 호출

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# --- 데이터 전처리 함수 ---
def preprocess_likert_data(df, columns):
    """
    - df: pandas DataFrame (원본 리커트 척도 데이터, 결측치는 np.nan)
    - columns: 처리할 컬럼 리스트

    리커트 점수, 결측치는 0으로 변환하여 tensor 반환
    """
    df_proc = df[columns].copy()

    # 결측치는 0으로 채우고, 관측값은 1~7 그대로 둠
    df_proc = df_proc.fillna(0).astype(int)

    # tensor 변환
    data_tensor = torch.LongTensor(df_proc.values)  # shape (N, num_vars)

    # 마스크: 결측=0, 관측=1 (mask dtype=torch.bool도 가능)
    mask = (data_tensor != 0).int()

    return data_tensor, mask

In [3]:
# --- CVAE 모델 ---
class CVAEOrdinal(nn.Module):
    #변수, 카테고리, 임베딩층, 잠재층, 은닉층
    # Removed default value for num_vars as it's passed during initialization
    # Changed order of arguments to fix SyntaxError
    def __init__(self, num_vars, num_categories, embedding_dim, latent_dim, hidden_dim):
        super().__init__()
        self.num_vars = num_vars
        self.num_categories = num_categories
        self.embedding_dim = embedding_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        # Use ModuleList for embeddings for each variable
        self.embeddings = nn.ModuleList([OrdinalEmbeddingLayer(num_categories, embedding_dim) for _ in range(num_vars)])

        # Encoder layers
        # Input to encoder is concatenated embeddings: num_vars * embedding_dim
        self.encoder_fc1 = nn.Linear(num_vars * embedding_dim, hidden_dim)
        self.encoder_mu = nn.Linear(hidden_dim, latent_dim)
        self.encoder_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder layers
        # Input to decoder is latent vector z: latent_dim
        self.decoder_fc1 = nn.Linear(latent_dim, hidden_dim)
        # Output of decoder: num_vars * num_categories (logits for each category per variable)
        self.decoder_output = nn.ModuleList([
            nn.Linear(hidden_dim, num_categories) for _ in range(num_vars)
        ])

    def encode(self, x):
        # x shape: (batch_size, num_vars)
        # Embed each variable separately and concatenate
        embedded = [self.embeddings[i](x[:, i]) for i in range(self.num_vars)] # embedded[i] shape: (batch_size, embedding_dim)
        embedded_cat = torch.cat(embedded, dim=1) # embedded_cat shape: (batch_size, num_vars * embedding_dim)

        # Pass through encoder FC layers
        h = F.relu(self.encoder_fc1(embedded_cat)) # h shape: (batch_size, hidden_dim)
        mu = self.encoder_mu(h)         # mu shape: (batch_size, latent_dim)
        logvar = self.encoder_logvar(h) # logvar shape: (batch_size, latent_dim)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        # Reparameterization trick
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std # z shape: (batch_size, latent_dim)

    def decode(self, z):
        # z shape: (batch_size, latent_dim)
        # Pass through decoder FC layers
        h = F.relu(self.decoder_fc1(z)) # h shape: (batch_size, hidden_dim)

        # Get output logits for each variable separately
        outputs = [self.decoder_output[i](h) for i in range(self.num_vars)] # outputs[i] shape: (batch_size, num_categories)
        return outputs # outputs is a list of num_vars tensors

    def forward(self, x):
        # x shape: (batch_size, num_vars)
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        outputs = self.decode(z)
        return outputs, mu, logvar # outputs is list of tensors, mu, logvar are tensors

In [4]:
# --- 손실 함수 (각 손실 리턴, beta 포함) ---
# target: original Likert values (1-7 or 0 for missing)
# outputs: list of logits (batch_size, num_categories) for each variable
# mask: (batch_size, num_vars), 1 for observed, 0 for missing
# beta: weighting factor for KL divergence
def loss_function(outputs, targets, mu, logvar, mask, beta):
    batch_size = targets.size(0)
    num_vars = targets.size(1)
    ce_loss = 0
    total_obs = 0 # Count of observed values in the batch

    # Calculate reconstruction loss only for observed values
    for i in range(num_vars):
        # Get indices where variable i is observed (mask is 1)
        obs_mask = (mask[:, i] == 1)
        obs_indices = obs_mask.nonzero(as_tuple=True)[0]

        if len(obs_indices) > 0:
            # Calculate Cross-Entropy loss for the i-th variable's output logits
            # and the corresponding target values (subtract 1 because CE expects 0-indexed labels)
            # outputs[i][obs_indices] shape: (num_observed_in_var_i, num_categories) (Logits)
            # targets[obs_indices, i] shape: (num_observed_in_var_i,) (Labels 1-7)
            ce_loss += F.cross_entropy(
                outputs[i][obs_indices],
                targets[obs_indices, i] - 1, # Subtract 1 to make labels 0-indexed (0-6)
                reduction='sum' # Sum the loss over observed values in this variable
            )
            total_obs += len(obs_indices) # Add number of observed values

    # Average reconstruction loss over ALL observed values in the batch
    if total_obs > 0:
        ce_loss = ce_loss / total_obs
    else:
        # If no observed values in the batch, reconstruction loss is 0
        ce_loss = torch.tensor(0.0, device=targets.device)


    # Calculate KL Divergence loss
    # D_KL(q(z|x) || p(z)) where p(z) is standard normal
    # -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    # Average KL loss over the batch
    kl_loss = kl_loss / batch_size

    # Total VAE loss with beta weighting
    total_loss = ce_loss + beta * kl_loss # Apply beta to KL loss

    return total_loss, ce_loss, kl_loss # Return individual loss components

In [5]:
# --- PyTorch Dataset 클래스 ---
class LikertDataset(Dataset):
    def __init__(self, data_tensor, mask):
        self.data = data_tensor
        self.mask = mask

    def __len__(self):
        return self.data.size(0)
    def __getitem__(self, idx):
        return self.data[idx], self.mask[idx]

# --- Ordinal Embedding Layer (using boundary points) ---
class OrdinalEmbeddingLayer(nn.Module):
    # Change the default value of num_categories from 7 to 5
    def __init__(self, num_categories=5, embedding_dim=1): # num_categories는 1-5 (5개), embedding_dim을 1로 설정
        super().__init__()
        self.num_categories = num_categories
        self.embedding_dim = embedding_dim # Should be 1 for this approach

        if embedding_dim != 1:
            print("Warning: OrdinalEmbeddingLayer with boundary points typically uses embedding_dim=1.")

        # Learnable boundary points. There are num_categories - 1 boundaries for num_categories categories.
        # For 5 categories (1, 2, 3, 4, 5), we need 4 boundaries.
        # The boundaries divide the 1D embedding space.
        self.boundaries = nn.Parameter(torch.randn(num_categories - 1)) # Learnable boundaries

        # Initialize boundaries to be somewhat ordered initially (optional but can help)
        # For example, sort them after initialization
        # self.boundaries.data.sort() # Can sort here or enforce constraint later

        # We still need an embedding for the value itself, which is projected onto the 1D space.
        # This could be a single shared parameter, or derived from context (though not in this simple layer).
        # A simple approach is to think of each observed score having a value on this 1D line.
        # Let's define learnable 'positions' for each category (1-5) on this 1D line.
        # There are num_categories positions.
        self.category_positions = nn.Parameter(torch.randn(num_categories, embedding_dim)) # Positions for categories 1 to num_categories on the 1D line


    def forward(self, x):
        # x shape: (batch_size,) containing values 0-5

        batch_size = x.size(0)
        output_embeddings = torch.zeros(batch_size, self.embedding_dim, device=x.device) # embedding_dim is 1

        # Identify non-missing values (indices > 0)
        observed_mask = (x > 0)
        observed_indices = x[observed_mask] # These values are 1-5

        if observed_indices.numel() > 0:
            # Get the learned positions for observed categories
            # Subtract 1 from observed_indices (1-5) to get 0-4 indices for category_positions
            output_embeddings[observed_mask] = F.embedding(observed_indices - 1, self.category_positions)

        # Missing values (index 0) will remain zero.

        # Note: The boundary points are used for the RECONSTRUCTION part of the VAE,
        # not directly in the embedding look-up here. The decoder will need to use
        # these boundaries to calculate the probability of each ordinal category.
        # This embedding layer now simply provides a 1D representation for each category.

        return output_embeddings

print("Modified OrdinalEmbeddingLayer: Using learnable category positions for 1D embedding.")

Modified OrdinalEmbeddingLayer: Using learnable category positions for 1D embedding.


#데이터호출

In [6]:
import pandas as pd

# 'your_folder/your_file.csv'를 Google Drive에 있는 실제 파일 경로로 바꾸세요.
file_path = '/content/drive/MyDrive/Projects/CVAE_MI/big5_100K.csv'
try:
    df = pd.read_csv(file_path) #, delimiter='\t')
    print("CSV 파일을 성공적으로 불러왔습니다:")
    #display(df.info())
    display(df.describe())

except FileNotFoundError:
    print(f"오류: 파일을 찾을 수 없습니다. 경로를 확인하세요: {file_path}")
except Exception as e:
    print(f"파일을 읽는 중 오류가 발생했습니다: {e}")


CSV 파일을 성공적으로 불러왔습니다:


,EXT1,EXT2,EXT3,EXT4,EXT5,EXT6,EXT7,EXT8,EXT9,EXT10,...,OPN1,OPN2,OPN3,OPN4,OPN5,OPN6,OPN7,OPN8,OPN9,OPN10
count,99996.000000,99996.000000,99996.000000,99996.000000,99996.00000,99996.000000,99996.000000,99996.000000,99996.000000,99996.000000,...,99996.000000,99996.000000,99996.000000,99996.000000,99996.000000,99996.000000,99996.000000,99996.000000,99996.000000,99996.000000
mean,2.661376,2.803352,3.298562,3.154256,3.25605,2.475839,2.787211,3.423757,2.991130,3.583433,...,3.618205,2.133195,3.979989,2.077143,3.749820,1.921747,3.953228,3.135325,4.066003,3.905406
std,1.253965,1.326333,1.210028,1.237583,1.27641,1.246493,1.390920,1.273129,1.347198,1.306076,...,1.143251,1.111266,1.102085,1.095002,0.992961,1.113772,1.006883,1.236958,1.055699,1.039489
min,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,2.000000,2.000000,2.000000,2.00000,2.000000,2.000000,2.000000,2.000000,3.000000,...,3.000000,1.000000,3.000000,1.000000,3.000000,1.000000,3.000000,2.000000,4.000000,3.000000
50%,3.000000,3.000000,3.000000,3.000000,3.00000,2.000000,3.000000,4.000000,3.000000,4.000000,...,4.000000,2.000000,4.000000,2.000000,4.000000,2.000000,4.000000,3.000000,4.000000,4.000000
75%,4.000000,4.000000,4.000000,4.000000,4.00000,3.000000,4.000000,4.000000,4.000000,5.000000,...,4.000000,3.000000,5.000000,3.000000,4.000000,2.000000,5.000000,4.000000,5.000000,5.000000
max,5.000000,5.000000,5.000000,5.000000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


#모델호출

In [7]:
import torch
import torch.nn as nn

# 모델을 저장했던 경로와 동일해야 합니다.
model_save_path = '/content/drive/MyDrive/Projects/CVAE_MI/model_0628.pt'

# 모델 아키텍처를 저장할 때와 동일하게 정의합니다.
# 필요한 파라미터(num_vars, num_categories, embedding_dim, latent_dim, hidden_dim)는
# 모델을 저장할 때 사용했던 값과 일치해야 합니다.
# 예시 파라미터 (이전 노트북 상태에서 사용된 값):
# num_vars=len(columns) (이는 실제 로드 시점에 df.columns의 길이와 일치해야 함)
# num_categories=5
# embedding_dim=1
# latent_dim=20
# hidden_dim=64

# Assuming 'df' with the correct columns is still available from previous steps
# If not, you might need to reload or recreate the DataFrame structure
# For this example, we'll assume 'df' is available and has the necessary columns.
num_vars = len(df.columns) # Get the number of variables from the current df
num_categories = 5
embedding_dim = 1
latent_dim = 64
hidden_dim = 128

# 디바이스 설정 (저장할 때 사용한 디바이스와 일치시킬 필요는 없지만, 일반적으로 모델 로드 후 사용할 디바이스를 설정)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 인스턴스 생성
loaded_model = CVAEOrdinal(num_vars=num_vars,
                           num_categories=num_categories,
                           embedding_dim=embedding_dim,
                           latent_dim=latent_dim,
                           hidden_dim=hidden_dim)

# 저장된 state_dict 로드
loaded_model.load_state_dict(torch.load(model_save_path, map_location=device))

# 모델을 설정한 디바이스로 이동
loaded_model.to(device)

# 모델을 평가 모드로 설정
loaded_model.eval()

print(f"모델이 '{model_save_path}' 경로에서 성공적으로 로드되었습니다.")
print(f"모델은 현재 {device} 디바이스에 있습니다.")

RuntimeError: Error(s) in loading state_dict for CVAEOrdinal:
	Missing key(s) in state_dict: "embeddings.0.boundaries", "embeddings.0.category_positions", "embeddings.1.boundaries", "embeddings.1.category_positions", "embeddings.2.boundaries", "embeddings.2.category_positions", "embeddings.3.boundaries", "embeddings.3.category_positions", "embeddings.4.boundaries", "embeddings.4.category_positions", "embeddings.5.boundaries", "embeddings.5.category_positions", "embeddings.6.boundaries", "embeddings.6.category_positions", "embeddings.7.boundaries", "embeddings.7.category_positions", "embeddings.8.boundaries", "embeddings.8.category_positions", "embeddings.9.boundaries", "embeddings.9.category_positions", "embeddings.10.boundaries", "embeddings.10.category_positions", "embeddings.11.boundaries", "embeddings.11.category_positions", "embeddings.12.boundaries", "embeddings.12.category_positions", "embeddings.13.boundaries", "embeddings.13.category_positions", "embeddings.14.boundaries", "em

#결측 제거

In [ ]:
# (df == 0) 각 요소가 0인지 여부 Bool DataFrame을 생성
# .any(axis=1) 각 행에 대해 열에 0이 있는지 확인
rows_with_zero = (df == 0).any(axis=1)

# 0을 포함하지 않는 행만 선택, 새로운 DataFrame을 생성
df = df[~rows_with_zero].copy()

print("원본 DataFrame의 행 수:", len(df))
print("0을 포함하는 행을 삭제한 후의 DataFrame 행 수:", len(df))

display(df.describe())

df_original = df.copy()

#결측 생성

In [ ]:
def introduce_missing_values(df, column_name, missingness_type, target_percentage):
    """
    Introduces missing values into a specified column of a DataFrame based on different missingness patterns.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to introduce missing values into.
        missingness_type (str): The type of missingness ('negative_uni',
                               'positive_uni', 'bidirection', or 'random').
        target_percentage (float): The desired percentage of missing values to introduce (e.g., 0.10 for 10%).

    Returns:
        pd.DataFrame: The DataFrame with missing values introduced in the specified column.
    """
    # Create a copy to avoid modifying the original DataFrame directly before applying NaNs
    df_modified = df.copy()

    # 1. Calculate the total number of values in the specified column
    total_values = df_modified[column_name].size

    # 2. Calculate the target number of missing values (10%) and round
    target_missing_count = int(round(total_values * target_percentage))

    # 3. Print the results
    # print(f"Total number of values in {column_name}: {total_values}")
    # print(f"Target number of missing values in {column_name} ({target_percentage*100:.0f}%): {target_missing_count}")

    # 4. Handle potential NaN values in the column before calculating probabilities or subsets by using `.dropna()`.
    # For 'random' missingness, we don't need to drop NaNs before selecting indices,
    # as we select from the entire column indices.
    if missingness_type != 'random':
        column_data_for_prob = df_modified[column_name].dropna()

        if column_data_for_prob.empty:
            print(f"No non-null values in {column_name} to introduce missingness.")
            return df_modified # Return df_modified as is

        # 5. Calculate the mean of the non-null values in the specified column.
        mean_value = column_data_for_prob.mean()

        # 6. Implement conditional logic based on the missingness_type:
        if missingness_type == 'negative_uni':
            # If missingness_type is 'negative_uni', select the subset of non-null values that are less than or equal to the mean.
            subset_for_missingness = column_data_for_prob[column_data_for_prob <= mean_value].copy()
            # Calculate the distance from the mean as mean_value - subset_value.
            distance_from_mean = mean_value - subset_for_missingness
        elif missingness_type == 'positive_uni':
            # If missingness_type is 'positive_uni', select the subset of non-null values that are greater than or equal to the mean.
            subset_for_missingness = column_data_for_prob[column_data_for_prob >= mean_value].copy()
            # Calculate the distance from the mean as subset_value - mean_value.
            distance_from_mean = subset_for_missingness - mean_value
        elif missingness_type == 'bidirection':
            # If missingness_type is 'bidirection', select all non-null values.
            subset_for_missingness = column_data_for_prob.copy()
            # Calculate the distance from the mean as the absolute difference: np.abs(subset_value - mean_value).
            distance_from_mean = np.abs(subset_for_missingness - mean_value)
        else:
            # If missingness_type is invalid, print an error message and return the original DataFrame.
            print(f"Error: Invalid missingness_type '{missingness_type}'. Use 'negative_uni', 'positive_uni', 'bidirection', or 'random'.")
            return df_modified # Return original df_modified if type is invalid

        # 7. If the selected subset for missingness is empty, print a message and return the original DataFrame.
        if subset_for_missingness.empty:
            print(f"No values in {column_name} match the criteria for missingness type '{missingness_type}' to introduce missingness.")
            return df_modified

        # 8. Calculate weights for each value in the selected subset based on the calculated distance from the mean.
        # Using distance + 1 to ensure positive weights and increasing probability with distance.
        weights = distance_from_mean + 1

        # 9. Normalize these weights to create selection probabilities for the subset.
        selection_probabilities = weights / weights.sum()

        # 10. Get the indices of the selected subset.
        subset_indices = subset_for_missingness.index

        # 11. Calculate the number of values to select from this subset to make missing, ensuring it does not exceed the size of the subset or the overall target missing count.
        num_to_select_from_subset = min(target_missing_count, len(subset_indices))

        if num_to_select_from_subset > 0:
            # 12. Use np.random.choice to randomly select the indices from the subset based on the calculated selection probabilities. Use replace=False to ensure unique indices.
            indices_to_make_missing = np.random.choice(
                subset_indices,
                size=num_to_select_from_subset,
                replace=False,  # Ensure unique indices are selected
                p=selection_probabilities  # Use calculated probabilities for selection
            )

            # 13. Introduce missing values (np.nan) into the specified column of the copied DataFrame at the selected indices using .loc[].
            df_modified.loc[indices_to_make_missing, column_name] = np.nan

            # 14. Print the number of indices selected to confirm the operation.
            # print(f"Attempted to introduce {num_to_select_from_subset} missing values in {column_name} based on criteria.")
            # print(f"Number of indices selected to make missing in {column_name}: {len(indices_to_make_missing)}")

        else:
            print(f"Target missing count is 0 or no values in {column_name} match the criteria for selection.")

    elif missingness_type == 'random':
         # If missingness_type is 'random', select indices randomly from the entire column
         # We need to ensure we don't try to make more values missing than available
         num_to_select_random = min(target_missing_count, total_values)

         if num_to_select_random > 0:
             # Get all indices in the column
             all_indices = df_modified[column_name].index

             # Randomly select indices to make missing
             indices_to_make_missing = np.random.choice(
                 all_indices,
                 size=num_to_select_random,
                 replace=False # Ensure unique indices are selected
             )

             # Introduce missing values (np.nan)
             df_modified.loc[indices_to_make_missing, column_name] = np.nan

            #  print(f"Attempted to introduce {num_to_select_random} random missing values in {column_name}.")
            #  print(f"Number of indices selected to make missing in {column_name}: {len(indices_to_make_missing)}")

         else:
             print(f"Target missing count is 0 or no values in {column_name} available to make missing.")


    # 15. Calculate the actual number of missing values in the column after introducing NaNs.
    actual_missing_count = df_modified[column_name].isnull().sum()

    # 16. Calculate the actual percentage of missing values and print it, formatted to two decimal places.
    actual_missing_percentage = (actual_missing_count / total_values) * 100
    # print(f"\nActual number of missing values in {column_name} after introduction: {actual_missing_count}")
    # print(f"Actual percentage of missing values in {column_name}: {actual_missing_percentage:.2f}%")

    # 17. Return the modified DataFrame.
    return df_modified

#결측 복원

In [ ]:
# --- 결측치 복원 함수 ---
def impute_missing_values(model, data_tensor_encoded, mask_tensor, device, num_categories):
    """
    model: 학습된 CVAEOrdinal 모델
    data_tensor_encoded: 결측치 0으로 인코딩된 데이터 tensor (N, num_vars)
    mask_tensor: 마스크 tensor (N, num_vars), 1 for observed, 0 for missing
    device: cpu or cuda
    num_categories: 리커트 척도 레벨 수 (5, 7 등)

    반환: imputed tensor (N, num_vars), int (1~7)
    """
    model.eval() # Set model to evaluation mode
    data_tensor_encoded = data_tensor_encoded.to(device)
    mask_tensor = mask_tensor.to(device)

    with torch.no_grad(): # Disable gradient calculation
        # Get model outputs (logits for each category)
        # In inference, we might pass data_tensor_encoded and mask_tensor to forward
        # But encode/decode structure is used here.
        # The encode part takes the data_tensor_encoded (with 0s)
        mu, logvar = model.encode(data_tensor_encoded)
        # For deterministic imputation, use mu; for stochastic imputation, sample from mu, logvar
        # Using mu for deterministic imputation here
        # z = mu # Use mean for deterministic
        # OR sample z for stochastic imputation:
        z = model.reparameterize(mu, logvar) # Use reparameterization for stochastic

        # Decode the latent representation to get logits
        # The decode function takes only z
        outputs = model.decode(z) # outputs is list of logits (N, num_categories)

        # Stack logits across variables: (N, num_vars, num_categories)
        logits_stacked = torch.stack(outputs, dim=1)

        # Get the predicted category (most probable Likert score)
        # argmax over the category dimension (dim=2)
        # Predicted categories are 0-indexed (0 to num_categories-1)
        predicted_categories = torch.argmax(logits_stacked, dim=2)

        # Convert 0-indexed categories back to 1-indexed Likert scores (1 to num_categories)
        imputed_vals = predicted_categories + 1 # shape (N, num_vars), values 1-7


        # Create the final imputed tensor:
        # Start with the original encoded tensor (which has 1-7 for observed, 0 for missing)
        imputed_tensor_final = data_tensor_encoded.clone()

        # Identify locations that were originally missing (mask is 0)
        # missing_mask shape: (N, num_vars) -> boolean
        missing_mask = (mask_tensor == 0)

        # Replace the values at missing locations in the final tensor with the imputed_vals
        imputed_tensor_final[missing_mask] = imputed_vals[missing_mask]


    return imputed_tensor_final.cpu() # Return tensor on CPU

#자동화

In [ ]:
import numpy as np
import pandas as pd
import torch # Ensure torch is imported

# Ensure the necessary functions and model are defined and available:
# preprocess_likert_data, impute_missing_values, loaded_model, OrdinalEmbeddingLayer, CVAEOrdinal

# Check if required components are available
required_components = ['preprocess_likert_data', 'impute_missing_values', 'loaded_model', 'OrdinalEmbeddingLayer', 'CVAEOrdinal']
for component in required_components:
    if component not in globals():
        print(f"Error: Required component '{component}' is not defined. Please ensure all necessary code cells have been run.")
        # You might want to add a 'break' or 'sys.exit()' here in a production script,
        # but for interactive Colab, printing an error and continuing might be acceptable
        # if some scenarios can still run (though likely not the imputation).
        # For this task, we must stop if core components are missing.
        raise NameError(f"Required component '{component}' is not defined.")


result = []
# Define the list of columns to introduce missingness and evaluate
columns_to_process = df_original.columns.tolist()

# Set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using device: {device}") # Removed verbose print

# Ensure the loaded_model is on the correct device
loaded_model.to(device)
loaded_model.eval() # Set model to evaluation mode


print("Starting evaluation across different missingness scenarios...")

# Loop through missingness types
for mis_type in ['random', 'negative_uni', 'positive_uni', 'bidirection']:
    # Loop through missingness rates
    for mis_rate in [0.10, 0.30, 0.50]:
        print(f"\nProcessing scenario: mis_type='{mis_type}', mis_rate={mis_rate:.2f}") # Keep this for progress tracking

        # Start with a fresh copy of the original dataframe for each scenario
        df_scenario = df_original.copy()

        # Introduce missing values column by column for the current mis_type and mis_rate
        # The introduce_missing_values function expects a target_percentage per column.
        for col in columns_to_process:
             df_scenario = introduce_missing_values(df_scenario, col, mis_type, mis_rate)

        # Removed verbose print of missing values per column
        # print(f"Missing values introduced. Current NaN count per column:")
        # display(df_scenario.isnull().sum())


        # --- Imputation Step ---
        # Prepare data for imputation using preprocess_likert_data
        data_tensor_encoded, mask_tensor = preprocess_likert_data(df_scenario, columns_to_process)
        # Removed verbose print
        # print("Data preprocessed for imputation.")

        # Perform imputation using the loaded model
        num_categories = 5 # Ensure this matches your model
        imputed_tensor = impute_missing_values(loaded_model, data_tensor_encoded, mask_tensor, device, num_categories)
        # Removed verbose print
        # print("Imputation complete.")

        # Convert imputed tensor back to a pandas DataFrame
        df_imputed_scenario = pd.DataFrame(imputed_tensor.cpu().numpy(), columns=columns_to_process, index=df_scenario.index)
        # Removed verbose print
        # print("Imputed tensor converted to DataFrame.")
        # display(df_imputed_scenario.head()) # Optional: display head of imputed data

        # --- Evaluation Step ---
        # Create a boolean mask for cells that are currently NaN in the df_scenario DataFrame
        current_missing_mask = df_scenario.isna()

        # Use df_original to get the true values at the locations that are currently missing
        df_original_aligned_for_eval = df_original.loc[df_scenario.index]

        # Extract original values for the cells that are currently missing
        original_missing_values_series = df_original_aligned_for_eval[current_missing_mask].stack()
        original_missing_values = original_missing_values_series.values

        # Extract imputed values for the cells that were missing in df_scenario
        imputed_values_for_missing_series = df_imputed_scenario[current_missing_mask].stack()
        imputed_values_for_missing = imputed_values_for_missing_series.values

        # Check if there are actually any missing values introduced and imputed in this scenario
        if len(original_missing_values) == 0 or len(imputed_values_for_missing) == 0:
            print(f"No missing values found for evaluation in scenario mis_type='{mis_type}', mis_rate={mis_rate:.2f}. Skipping metrics calculation.")
            result.append({
                'mis_type': mis_type,
                'mis_rate': mis_rate,
                'accuracy': np.nan,
                'mae': np.nan,
                'rmse': np.nan
            })
            continue

        if len(original_missing_values) != len(imputed_values_for_missing):
            print(f"Error: Mismatch in number of original missing ({len(original_missing_values)}) and imputed ({len(imputed_values_for_missing)}) values for scenario mis_type='{mis_type}', mis_rate={mis_rate:.2f}. Skipping metrics calculation.")
            result.append({
                'mis_type': mis_type,
                'mis_rate': mis_rate,
                'accuracy': np.nan,
                'mae': np.nan,
                'rmse': np.nan
            })
            continue

        # Removed verbose print of calculation info
        # print(f"Calculating metrics for {len(original_missing_values)} imputed values...")

        # Convert to numpy arrays for easier calculation and ensure they are integers
        original_missing_values = original_missing_values.astype(int)
        imputed_values_for_missing = imputed_values_for_missing.astype(int)

        # Calculate Imputation Accuracy
        accuracy = np.mean(original_missing_values == imputed_values_for_missing)

        # Calculate Mean Absolute Error (MAE)
        mae = np.mean(np.abs(original_missing_values - imputed_values_for_missing))

        # Calculate Root Mean Squared Error (RMSE)
        rmse = np.sqrt(np.mean((original_missing_values - imputed_values_for_missing)**2))

        # Append the calculated metrics to the result list
        result.append({
            'mis_type': mis_type,
            'mis_rate': mis_rate,
            'accuracy': accuracy,
            'mae': mae,
            'rmse': rmse
        })

        # Removed verbose print
        # print(f"Completed scenario: mis_type='{mis_type}', mis_rate={mis_rate:.2f}")


print("\n--- Evaluation Complete ---")

# Convert results to a DataFrame for better readability
results_df = pd.DataFrame(result)

# Display the results summary
print("\nEvaluation Results Summary:")
display(results_df)